# Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 
# pour afficher facilement les graphiques 
from urllib.request import urlopen
from bs4 import BeautifulSoup
import xml.etree.ElementTree as ET
import os
import re
from pathlib import Path
from nltk.stem import SnowballStemmer
import spacy
from correction_requetetd4 import corriger_requete


# Variables 

In [ ]:
# configuration des chemins 
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd().parent

BULLETINS = BASE_DIR / "BULLETINS"
DATA = BASE_DIR / "data"
OUTPUT = BASE_DIR / "output"

# Variables 

In [ ]:
def extraction_metadonnees(requete):
    metadonnees = {}
    # contriantes temporelle
    pattern_date =  r"entre le \d{1,2} \w+ \d{4} et \d{1,2} \w+ \d{4}|" \
                    r"entre \d{1,2}/\s*\d{1,2}/\s*\d{4} et \d{1,2}/\s*\d{1,2}/\s*\d{4}|"\
                    r"entre \d{4} et \d{4}|" \
                    r"après \d{1,2}/\s*\d{1,2}/\s*\d{4} |"\
                    r"de \d{4} |" \
                    r"au mois de \w+ \d{4}|"\
                    r"du mois de \w+ \d{4}|"\
                    r"au mois de \w+ |"\
                    r"en \w+ \d{4}|"\
                    r"en \d{4}|"\
                    r"à partir de \d{4}|"\
                    r"à partir de \w+ \d{4}|"\
                    r"de l'année \d{4}|"\
                    r"après janvier \d{4}|"\
                    r"du \d{1,2} \w+ \d{4}"\

    resultat_date = re.findall(pattern_date,  requete, re.IGNORECASE)
    metadonnees['dates'] = resultat_date[0] if len(resultat_date) > 0 else ""

    # rubrique 
    pattern_rubrique = r"(focus|horizons enseignement|en direct des laboratoires|a lire|actualités innovations|événement)"
    resultat_rubrique = re.findall(pattern_rubrique, requete, re.IGNORECASE)
    metadonnees['rubrique'] = resultat_rubrique

    # filtre structurel
    pattern_structurel = r"avec des images|contenant une imagesans|qui \w+ des images|sans image|contenant le mot \w+"
    resultat_structurel = re.findall(pattern_structurel,  requete, re.IGNORECASE)
    metadonnees['structurel'] = resultat_structurel[0] if len(resultat_structurel) > 0 else ""

    #operateur
    pattern_operateur = r"et | ou |mais pas | sans "
    resultat_operateur = re.findall(pattern_operateur,  requete, re.IGNORECASE)
    metadonnees['operateur'] = list(set(resultat_operateur))

    return metadonnees  


In [74]:
requete = " Chercher les articles dans le domaine industriel et datés à partir de 2012"
dic = extraction_metadonnees(requete)
print(dic)

{'dates': '', 'rubrique': [], 'structurel': '', 'operateur': ['et ']}


In [51]:
def reste(requette, metadonnees):
    mots_restant = requette
    for data in metadonnees :
        data = metadonnees[data]
        if isinstance(data,list):
            for element in data :
                mots_restant = mots_restant.replace(element,"")
        else :
            mots_restant = mots_restant.replace(data,"")

    return mots_restant 

In [52]:
rst = reste(requete, dic)
print(rst)

Je veux les articles de la rubrique  parlant de la santé 


In [ ]:
mois_dict = {
    "janvier": "01", "février": "02", "mars": "03",
    "avril": "04", "mai": "05", "juin": "06",
    "juillet": "07", "août": "08", "septembre": "09",
    "octobre": "10", "novembre": "11", "décembre": "12"
}

def convertir_date(date_str):
    """
    transforme 10 avril 2012 en 10/04/2012
    
    """
    parts = date_str.split()

    jour = parts[0].zfill(2)
    mois_str = mois_dict[parts[1]]
    annee = parts[2]

    return f"{jour}/{mois_str}/{annee}"

def traiter_expression(expr):
    """
    Le but de la fonction c'est d'avoir des dates in et out pour l'obtention de l'intervale sous un format datetime comprehensible par python 
    """
    dates = {}
    pattern = r"entre le (\d{1,2} \w+ \d{4}) et (\d{1,2} \w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = convertir_date(match.group(1))
        date2 = convertir_date(match.group(2))
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"entre( \d{1,2}/\s*\d{1,2}/\s*\d{4}) et (\d{1,2}/\s*\d{1,2}/\s*\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = match.group(2)
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"entre (\d{4}) et (\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = match.group(2)
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"après (\d{1,2}/\s*\d{1,2}/\s*\d{4}) "
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = None
        dates['in'] = date1
        dates['out'] = date2
    
    pattern = r"à partir de (\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = match.group(1)
        date2 = None
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"au mois de (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"du mois de (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"en (\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = f"01/{int(mois_min)+1}/{match.group(1).split()[1]}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"(?:en|de| de l'année)\s+(\d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = f"01/01/{match.group(1)}"
        date2 = f"31/12/{match.group(1)}"
        dates['in'] = date1
        dates['out'] = date2

    pattern = r"(?:à partir de | après )\s+(\w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        mois_min = mois_dict[match.group(1).split()[0]]
        date1 = f"01/{int(mois_min)}/{match.group(1).split()[1]}"
        date2 = None
        dates['in'] = date1
        dates['out'] = date2
    
    pattern = r"du (\d{1,2} \w+ \d{4})"
    match = re.match(pattern, expr)
    if match:
        date1 = convertir_date(match.group(1))
        date2 = convertir_date(match.group(1))
        dates['in'] = date1
        dates['out'] = date2
    
    return dates



In [67]:
expr = "au mois de avril 2012"
output = traiter_expression(expr)
print(output)

01/04/2012 01/5/2012
{'in': '01/04/2012', 'out': '01/5/2012'}


In [79]:
def representation_structuree(requette):
    metadonnees = extraction_metadonnees(requette)
    mots_restant = reste(requette, metadonnees)
    mots_cles = mots_restant.split()
    dates = traiter_expression(metadonnees['dates'])

    structuration = {
        'mots_cles': mots_cles,
        'rubrique' : metadonnees['rubrique'], 
        'operateur' : [op.upper() for op in metadonnees['operateur']],
        'date_min' : dates['in'], 
        'date_max' : dates['out']
    }
    return structuration

In [80]:
requete = "Jevoudraisles articles qui datent du 1 décembre 2012 et dont la rubrique est Actualités Innovations"
representation_structuree(requete)

01/12/2012 01/12/2012


{'mots_cles': ['Jevoudraisles',
  'articles',
  'qui',
  'datent',
  'dont',
  'la',
  'rubrique',
  'est'],
 'rubrique': ['Actualités Innovations'],
 'operateur': ['ET '],
 'date_min': '01/12/2012',
 'date_max': '01/12/2012'}